# 📊 CERT Dataset — Exploratory Data Analysis

Interactive walkthrough of the CERT r4.2 insider threat dataset features.

**Dataset**: 200 employees, 90 days, 211 enhanced features, ~11,500 employee-day records.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import sys, os

# Setup
ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(ROOT))
os.chdir(str(ROOT))

sns.set_theme(style='darkgrid', palette='coolwarm')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.family': 'sans-serif'})

print(f'Project root: {ROOT}')

## 1. Load Data

In [ ]:
from argus.config import Config
Config.setup()

data_dir = Config.paths.DATA / 'processed'
X = np.load(data_dir / 'X_enhanced.npy')
y = np.load(data_dir / 'y_enhanced.npy')
feature_names = json.loads((data_dir / 'enhanced_feature_cols.json').read_text())

# If 3D sequences, take last timestep
if X.ndim == 3:
    print(f'Sequence data: {X.shape} -> using last timestep')
    X = X[:, -1, :]

df = pd.DataFrame(X, columns=feature_names)
df['is_insider'] = y.astype(int)

print(f'Shape: {df.shape}')
print(f'Insiders: {int(y.sum())} / {len(y)} ({y.mean()*100:.1f}%)')
df.head()

## 2. Feature Distributions — Insiders vs Normal

In [ ]:
# Key discriminative features
key_features = ['login_hour', 'data_volume_mb', 'files_accessed', 'usb_events',
                'unique_systems_accessed', 'external_email_ratio',
                'temporal_entropy', 'clearance_normalized']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for i, feat in enumerate(key_features):
    ax = axes.flatten()[i]
    if feat in df.columns:
        df[df['is_insider']==0][feat].hist(ax=ax, bins=50, alpha=0.6, label='Normal', color='#06b6d4', density=True)
        df[df['is_insider']==1][feat].hist(ax=ax, bins=50, alpha=0.6, label='Insider', color='#ef4444', density=True)
        ax.set_title(feat.replace('_', ' ').title(), fontsize=11)
        ax.legend(fontsize=8)
plt.suptitle('Feature Distributions: Normal vs Insider', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Correlation Heatmap — Base Features

In [ ]:
base_feats = [f for f in feature_names if not f.startswith(('roll_', 'expanding_', 'delta_', 'abs_delta_', 'zscore_'))]
corr = df[base_feats].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title('Base Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Class Separation — PCA Visualization

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_clean = np.nan_to_num(X, 0.0)
X_scaled = StandardScaler().fit_transform(X_clean)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 8))
scatter_normal = ax.scatter(X_pca[y==0, 0], X_pca[y==0, 1], c='#06b6d4', alpha=0.3, s=10, label='Normal')
scatter_insider = ax.scatter(X_pca[y==1, 0], X_pca[y==1, 1], c='#ef4444', alpha=0.8, s=40, marker='x', label='Insider', linewidths=2)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=12)
ax.set_title('PCA: 211 Features Projected to 2D', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f'Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%')

## 5. Feature Category Statistics

In [ ]:
categories = {
    'Base Features': [f for f in feature_names if not any(f.startswith(p) for p in ['roll_', 'expanding_', 'delta_', 'abs_delta_', 'zscore_', 'clearance'])],
    'Clearance': [f for f in feature_names if 'clearance' in f],
    'Rolling 7d': [f for f in feature_names if f.startswith('roll_7d_')],
    'Rolling 14d': [f for f in feature_names if f.startswith('roll_14d_')],
    'Expanding': [f for f in feature_names if f.startswith('expanding_')],
    'Deltas': [f for f in feature_names if f.startswith(('delta_', 'abs_delta_'))],
    'Z-Scores': [f for f in feature_names if f.startswith('zscore_')],
}

summary = []
for cat, feats in categories.items():
    if feats:
        sub = df[feats]
        summary.append({
            'Category': cat,
            'Count': len(feats),
            'Mean NaN%': sub.isna().mean().mean() * 100,
            'Mean Std': sub.std().mean(),
        })

pd.DataFrame(summary).style.format({'Mean NaN%': '{:.2f}%', 'Mean Std': '{:.3f}'}).set_caption('Feature Category Summary')

## 6. Summary

| Property | Value |
|----------|-------|
| Employees | 200 |
| Duration | 90 days |
| Total records | ~11,500 employee-days |
| Base features | 47 |
| Enhanced features | 211 |
| Insider rate | 2.9% (333/11,510) |
| Top discriminative feature | `clearance_normalized` (SHAP=0.610) |